# Pipeline d'Inférence et d'Analyse LAMNr Flows (2D)
**Projet :** Modélisation Latente Multimodale (T1 / T2 / FA)

Ce notebook exécute le flux de travail post-entraînement complet pour les modèles LAMNr Flows 2D. Il gère l'estimation de la distribution (ajustement Gaussien), la traduction entre modalités (imputation), la synthèse de templates, et la manipulation avancée de l'espace latent (interpolation, scaling).

In [ ]:
import os

# =============================================================================
# CONFIGURATION GLOBALE
# =============================================================================

# Définition des chemins principaux
BASE_DIR = "/Users/ntustison/Desktop/lamnr_glow_dlbs"
WHICH_EXPERIMENT = "96x128"
RUNS_DIR = f"{BASE_DIR}/runs2d/dlbs_t1_t2flair_fa_{WHICH_EXPERIMENT}_K12_L5_HC192"
OUT_DIR = f"{BASE_DIR}/output{WHICH_EXPERIMENT}"
MANIFEST_DIR = f"{BASE_DIR}/manifests"

# Fichiers spécifiques
CKPT = f"{RUNS_DIR}/training_state.pt"
MANIFEST = f"{MANIFEST_DIR}/manifest.csv"
MANIFEST_SHORT = f"{MANIFEST_DIR}/manifest_short.csv"
MANIFEST_LESIONS = f"{MANIFEST_DIR}/manifest_brats_short.csv"

# Images de référence
ANTS_TEMPLATE = "/Users/ntustison/Data/Public/OpenNeuro/ds004856/Template/nki_x.nii.gz"
EXAMPLE_IMAGE = "/Users/ntustison/Data/Public/OpenNeuro/ds004856/BIDSAlignedToTemplate/sub-1022/ses-wave1/anat/sub-1022_ses-wave1_acq-MPRAGE_run-1_T1w.nii.gz"

# Paramètres d'exécution
SLICE_INDEX = 115
DEVICE = "cpu"
WHICH_PYTHON = "/Users/ntustison/anaconda3/bin/python3"

# Modèles dérivés
GAUSSIAN_LR = f"{OUT_DIR}/t1_t2_fa_lowrank.npz"
GAUSSIAN_LR_SUMMARY = f"{OUT_DIR}/t1_t2_fa_lowrank_summary.json"
DIST_CSV = f"{OUT_DIR}/t1_distance_to_gaussian.csv"

# Création du répertoire de sortie
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Répertoire de travail configuré : {OUT_DIR}")

In [ ]:
## 1. Ajustement du Modèle Gaussien (Gauss-Fit)
**Objectif :** Estimer la moyenne multivariée et la matrice de covariance de l'espace latent (prior). L'estimateur `lowrank` (SVD - Décomposition en Valeurs Singulières) est utilisé pour approximer la covariance sans saturer la mémoire RAM.

In [ ]:
if not os.path.exists(GAUSSIAN_LR):
    print("Ajustement de la distribution Gaussienne (Low-Rank SVD)...")
    !{WHICH_PYTHON} lamnr_glow_tool.py gauss-fit \
        --ckpt {CKPT} \
        --manifest {MANIFEST} \
        --views T1,T2,FA \
        --slice-axis 2 --slice-index {SLICE_INDEX} \
        --batch 64 --devices {DEVICE} \
        --cov-mode perlevel \
        --cov-estimator full \
        --rank 256 \
        --gauss-out {GAUSSIAN_LR} \
        --gauss-summary {GAUSSIAN_LR_SUMMARY}
else:
    print("Modèle Gaussien déjà existant. Étape ignorée.")

## 2. Exportation des Coupes 2D (Slicing)
**Objectif :** Extraire la coupe axiale définie par `SLICE_INDEX` depuis les volumes NIfTI 3D. Prépare les tenseurs exacts vus par le modèle pour inspection visuelle.

In [ ]:
MANIFEST_INPUT_DIR = f"{OUT_DIR}/manifest_input/"

if not os.path.exists(MANIFEST_INPUT_DIR):
    os.makedirs(MANIFEST_INPUT_DIR, exist_ok=True)
    print("Exportation des coupes 2D depuis les volumes 3D...")
    !{WHICH_PYTHON} lamnr_glow_tool.py export-slices \
        --manifest {MANIFEST_SHORT} \
        --slice-axis 2 --slice-index {SLICE_INDEX} \
        --views T1,T2,FA \
        --image-size {WHICH_EXPERIMENT} \
        --outdir {MANIFEST_INPUT_DIR} \
        --output-format nii.gz
else:
    print("Le répertoire d'exportation existe déjà. Étape ignorée.")

## 3. Imputation Gaussienne (Traduction Cross-Modale)
**Objectif :** Prédire une modalité manquante (ex: T1) à partir de modalités observées (ex: T2, FA). Le modèle utilise l'identité de Woodbury pour calculer la moyenne conditionnelle exacte (MMSE) dans l'espace latent.

In [ ]:
# 3A. T2 -> T1
impute_out_dir = f"{OUT_DIR}/impute_T1_from_T2/"
if not os.path.exists(impute_out_dir) or not os.listdir(impute_out_dir):
    os.makedirs(impute_out_dir, exist_ok=True)
    !{WHICH_PYTHON} lamnr_glow_tool.py gauss-impute \
        --ckpt {CKPT} --gauss {GAUSSIAN_LR} --manifest {MANIFEST_SHORT} \
        --views T1,T2,FA --observed T2 --target T1 \
        --slice-axis 2 --slice-index {SLICE_INDEX} \
        --batch 2 --devices {DEVICE} --outdir "{impute_out_dir}" --output-format nii.gz

# 3B. FA -> T1, T2
impute_out_dir = f"{OUT_DIR}/impute_T1T2_from_FA/"
if not os.path.exists(impute_out_dir) or not os.listdir(impute_out_dir):
    os.makedirs(impute_out_dir, exist_ok=True)
    !{WHICH_PYTHON} lamnr_glow_tool.py gauss-impute \
        --ckpt {CKPT} --gauss {GAUSSIAN_LR} --manifest {MANIFEST_SHORT} \
        --views T1,T2,FA --observed FA --target T1,T2 \
        --slice-axis 2 --slice-index {SLICE_INDEX} \
        --batch 2 --devices {DEVICE} --outdir "{impute_out_dir}" --output-format nii.gz

# 3C. T2, FA -> T1
impute_out_dir = f"{OUT_DIR}/impute_T1_from_T2FA/"
if not os.path.exists(impute_out_dir) or not os.listdir(impute_out_dir):
    os.makedirs(impute_out_dir, exist_ok=True)
    !{WHICH_PYTHON} lamnr_glow_tool.py gauss-impute \
        --ckpt {CKPT} --gauss {GAUSSIAN_LR} --manifest {MANIFEST_SHORT} \
        --views T1,T2,FA --observed T2,FA --target T1 \
        --slice-axis 2 --slice-index {SLICE_INDEX} \
        --batch 2 --devices {DEVICE} --outdir "{impute_out_dir}" --output-format nii.gz

## 4. Vérification de Reconstruction (Sanity Check)
**Objectif :** Confirmer l'inversibilité parfaite du flux ($x \leftrightarrow z$). Génère une grille comparative : Original | Reconstruit | Erreur Absolue.

In [ ]:
recon_panel_out = f"{OUT_DIR}/recon_panel.png"
if not os.path.exists(recon_panel_out):
    print("Génération du panneau de vérification de reconstruction...")
    !{WHICH_PYTHON} lamnr_glow_tool.py recon \
        --ckpt {CKPT} --manifest {MANIFEST_SHORT} --views T1,T2,FA --view-index 0 \
        --slice-axis 2 --slice-index {SLICE_INDEX} --batch 6 --devices {DEVICE} \
        --out {recon_panel_out} 
else:
    print("Le panneau de reconstruction existe déjà. Étape ignorée.")
    
# Affichage optionnel dans le notebook
from IPython.display import Image, display
display(Image(filename=recon_panel_out))

## 5. Échantillonnage Stochastique (Génération)
**Objectif :** Tirer des vecteurs aléatoires depuis la distribution normale et les décoder. La température (variance) module la netteté et la diversité anatomique.

In [ ]:
grid_size = "5x5"
modalities = {"t1": 0, "t2": 1, "fa": 2}
temperatures = [0.01, 0.25, 0.50, 0.75, 1.00]

for mod, v_idx in modalities.items():
    for temp in temperatures:
        sample_output = f"{OUT_DIR}/Samples/samples_{mod}_temp_{temp}.png"
        if not os.path.exists(sample_output):
            os.makedirs(os.path.dirname(sample_output), exist_ok=True)
            print(f"Échantillonnage ({mod}) à température = {temp}...")
            !{WHICH_PYTHON} lamnr_glow_tool.py sample \
                --ckpt {CKPT} --view-index {v_idx} --sample-grid-size {grid_size} \
                --image-size {WHICH_EXPERIMENT} --temperature {temp} \
                --devices {DEVICE} --sample-grid-out "{sample_output}" --seed 42

## 6. Reconstruction de Template (Atlas de Population)
**Objectif :** Décoder le vecteur latent moyen ($\mu$). L'échantillonnage de Monte-Carlo (`--mc-samples`) ajoute une micro-variance moyennée pour obtenir un atlas extrêmement net et dépourvu de bruit haute fréquence.

In [ ]:
output_template = f"{OUT_DIR}/template_T1_mu_sharpened.png"
if not os.path.exists(output_template):
    print("Génération du template de population (Moyenne Latente)...")
    !{WHICH_PYTHON} lamnr_glow_tool.py recon-template \
        --ckpt {CKPT} --gauss {GAUSSIAN_LR} --views T1 --view-index 0 \
        --mc-samples 10 --mc-temp 0.01 --out "{output_template}" \
        --sharpen-image --devices {DEVICE} --seed 42
else:
    print("Le template de population existe déjà. Étape ignorée.")

## 7. Interpolation Latente (Morphing Géodésique)
**Objectif :** Calculer une trajectoire linéaire dans l'espace latent entre deux cerveaux, produisant un morphing non-linéaire (anatomiquement continu) dans l'espace de l'image via interpolation sphérique (`slerp`).

In [ ]:
t_values = [0.00, 0.25, 0.50, 0.75, 1.00]

# 7A. Sujet -> Cerveau Moyen
for t_val in t_values:
    output_interp = f"{OUT_DIR}/interpolation/interp_mean_t{t_val}.nii.gz"
    if not os.path.exists(output_interp):
        os.makedirs(os.path.dirname(output_interp), exist_ok=True)
        !{WHICH_PYTHON} lamnr_glow_tool.py recon-interpolate \
            --ckpt {CKPT} --gauss {GAUSSIAN_LR} --source-image {EXAMPLE_IMAGE} \
            --views T1 --slice-axis 2 --slice-index {SLICE_INDEX} --devices {DEVICE} \
            --t {t_val} --interp-level 0,1.0 --interp-level 1,1.0 --out "{output_interp}"

# 7B. Sujet -> Template ANTs Cible
for t_val in t_values:
    output_interp = f"{OUT_DIR}/interpolation/inter_dlbs_example_t{t_val}.nii.gz"
    if not os.path.exists(output_interp):
        os.makedirs(os.path.dirname(output_interp), exist_ok=True)
        !{WHICH_PYTHON} lamnr_glow_tool.py recon-interpolate \
            --ckpt {CKPT} --gauss {GAUSSIAN_LR} --source-image {ANTS_TEMPLATE} \
            --target-image {EXAMPLE_IMAGE} --views T1 \
            --slice-axis 2 --slice-index {SLICE_INDEX} --devices {DEVICE} \
            --t {t_val} --out "{output_interp}"

## 8. Distances Latentes (Détection d'Anomalies)
**Objectif :** Calculer la distance géodésique ou euclidienne de chaque sujet par rapport au centre de la distribution. Permet d'isoler les sujets atypiques (outliers).

In [ ]:
if not os.path.exists(DIST_CSV):
    print("Calcul des distances latentes par rapport au modèle Gaussien...")
    !{WHICH_PYTHON} lamnr_glow_tool.py calc-distance \
        --ckpt {CKPT} --gauss {GAUSSIAN_LR} --manifest {MANIFEST} \
        --views T1 --slice-axis 2 --slice-index {SLICE_INDEX} \
        --out "{DIST_CSV}" --distance-metric geodesic --devices {DEVICE} --save-levels 
else:
    print("Le fichier CSV de distances existe déjà. Étape ignorée.")

## 9. Mise à l'échelle par Température (Temperature Scaling)
**Objectif :** Contracter l'espace latent ($\tau < 1.0$) pour forcer une image pathologique à se rapprocher du manifold sain. L'application par niveau cible des fréquences spécifiques (L0 = micro-structures, L5 = macro-géométrie).

In [ ]:
taus = [0.01, 0.25, 0.50, 0.75, 0.95, 0.99]

# 9A. Scaling L0 (Micro-structures)
for tau in taus:
    output_temp = f"{OUT_DIR}/temperature/recon_temperature_L0_tau{tau}.nii.gz"
    if not os.path.exists(output_temp):
        os.makedirs(os.path.dirname(output_temp), exist_ok=True)
        !{WHICH_PYTHON} lamnr_glow_tool.py recon-temperature \
            --ckpt {CKPT} --manifest {MANIFEST_LESIONS} --views T1 \
            --slice-axis 2 --slice-index {SLICE_INDEX} --devices {DEVICE} \
            --out "{output_temp}" --tau-level 0,{tau}

# 9B. Scaling L5 (Macro-structure globale)
for tau in taus:
    output_temp = f"{OUT_DIR}/temperature/recon_temperature_L5_tau{tau}.nii.gz"
    if not os.path.exists(output_temp):
        os.makedirs(os.path.dirname(output_temp), exist_ok=True)
        !{WHICH_PYTHON} lamnr_glow_tool.py recon-temperature \
            --ckpt {CKPT} --manifest {MANIFEST_LESIONS} --views T1 \
            --slice-axis 2 --slice-index {SLICE_INDEX} --devices {DEVICE} \
            --out "{output_temp}" --tau-level 5,{tau}

# 9C. Scaling Global
for tau in taus:
    output_temp = f"{OUT_DIR}/temperature/recon_temperature_Global_tau{tau}.nii.gz"
    if not os.path.exists(output_temp):
        os.makedirs(os.path.dirname(output_temp), exist_ok=True)
        !{WHICH_PYTHON} lamnr_glow_tool.py recon-temperature \
            --ckpt {CKPT} --manifest {MANIFEST_LESIONS} --views T1 \
            --slice-axis 2 --slice-index {SLICE_INDEX} --devices {DEVICE} \
            --out "{output_temp}" --tau {tau}

print("Pipeline 2D terminé.")